## Import libraries and general setup
In order for the agents to work, LM Studio must be running on `http://localhost:1234/v1

In [17]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from typing import TypedDict, Dict, Any, Optional, List
from IPython.display import display, Markdown
import warnings, json
from langchain_openai import ChatOpenAI
from langchain_core.callbacks import StreamingStdOutCallbackHandler


#two separate instances are needed: llm uses streaming=True for real-time text generation, while llm_json uses streaming=False to return a single complete block parsable as JSON.
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="gemma-3-4b-it",
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

llm_json = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="gemma-3-4b-it",
    temperature=0,
    streaming=False,
)

#In order to be sure that LM studio is working (because it happened many times that some of us forgot to turn it on) we added this line to check if it's running
test_response = llm.invoke("Say OK if you are working.").content
print(f"\n\nLLM test passed: {test_response[:50]}")

OK


LLM test passed: OK



---

## 1 — Data Agent

The Data Agent is in charge of loading the two datasets, `ALLARMI.csv` e `TIPOLOGIA_VIAGGIATORE.csv`, clean them (types, missing values, normalization) and filters by user perimeter.
It utilizes the LLM to generate a Data Quality Assessment that identifies issues and suggests corrective actions.

In [18]:
class DataAgent:

    def __init__(self, allarmi_path: str, tipologia_path: str):
        self.allarmi_path = allarmi_path
        self.tipologia_path = tipologia_path

    def _clean_allarmi(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        #conversion of numerical columns
        df['TOT'] = pd.to_numeric(df['TOT'], errors='coerce').fillna(0).astype(int)
        df['3zona'] = pd.to_numeric(df['3zona'], errors='coerce').fillna(-1).astype(int)
        df['ANNO_PARTENZA'] = pd.to_numeric(df['ANNO_PARTENZA'], errors='coerce').fillna(0).astype(int)
        df['MESE_PARTENZA'] = pd.to_numeric(df['MESE_PARTENZA'], errors='coerce').fillna(0).astype(int)

        #parsing date
        df['DATA_PARTENZA'] = pd.to_datetime(df['DATA_PARTENZA'], errors='coerce')

        #cleaning of str columns
        str_cols = ['AREOPORTO_ARRIVO', 'AREOPORTO_PARTENZA', 'PAESE_ARR',
                    'PAESE_PART', 'MOTIVO_ALLARME', 'OCCORRENZE', 'ZONA']
        for col in str_cols:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()

        #normalization of strange values in ZONA and flag_rischio
        df['ZONA'] = df['ZONA'].replace({'??': '-1', 'nan': '-1'})
        df['ZONA'] = pd.to_numeric(df['ZONA'], errors='coerce').fillna(-1).astype(int)
        df['flag_rischio'] = df['flag_rischio'].fillna('N/A').astype(str).str.strip()

        #feature engineering of the route (ROTTA) columns
        df['ROTTA'] = df['AREOPORTO_PARTENZA'] + ' → ' + df['AREOPORTO_ARRIVO']

        return df

    def _clean_tipologia(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        #conversion of numerical columns
        for col in ['ENTRATI', 'INVESTIGATI', 'ALLARMATI']:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
        df['GIORNO_PARTENZA'] = pd.to_numeric(df['GIORNO_PARTENZA'], errors='coerce').fillna(0).astype(int)
        df['ANNO_PARTENZA'] = pd.to_numeric(df['ANNO_PARTENZA'], errors='coerce').fillna(0).astype(int)
        df['MESE_PARTENZA'] = pd.to_numeric(df['MESE_PARTENZA'], errors='coerce').fillna(0).astype(int)

        #parsing date
        df['DATA_PARTENZA'] = pd.to_datetime(df['DATA_PARTENZA'], errors='coerce')

        #normalization of genre
        df['GENERE'] = df['GENERE'].astype(str).str.strip().str.upper()
        df['GENERE'] = df['GENERE'].replace({'NAN': 'N.D.', '-': 'N.D.'})

        #normalization of age band (FASCIA_ETA)
        df['FASCIA_ETA'] = df['FASCIA_ETA'].astype(str).str.strip()
        df['FASCIA_ETA'] = df['FASCIA_ETA'].replace(
            {'N/C': 'N.D.', 'minore': '0-17', 'adulto': '31-45', '-5': 'N.D.', 'nan': 'N.D.'}
        )

        #normalization of document type (TIPO_DOCUMENTO)
        df['TIPO_DOCUMENTO'] = df['TIPO_DOCUMENTO'].astype(str).str.strip()
        df['TIPO_DOCUMENTO'] = df['TIPO_DOCUMENTO'].replace(
            {'?': 'N.D.', '-': 'N.D.', ' ': 'N.D.', 'nan': 'N.D.', 'n.d.': 'N.D.'}
        )

        #normalization of nationality (NAZIONALITA) and check outcome (ESITO_CONTROLLO)
        df['NAZIONALITA'] = df['NAZIONALITA'].astype(str).str.strip().str.upper()
        df['NAZIONALITA'] = df['NAZIONALITA'].replace({'NAN': 'N.D.', '': 'N.D.'})
        df['ESITO_CONTROLLO'] = df['ESITO_CONTROLLO'].fillna('N.D.').astype(str).str.strip()

        #feature engineering of the route (ROTTA) and rates (TASSO_ALLARME and TASSO_INVESTIGAZIONE) columns
        df['ROTTA'] = df['AREOPORTO_PARTENZA'] + ' → ' + df['AREOPORTO_ARRIVO']
        df['TASSO_ALLARME'] = np.where(df['ENTRATI'] > 0, df['ALLARMATI'] / df['ENTRATI'], 0.0)
        df['TASSO_INVESTIGAZIONE'] = np.where(df['ENTRATI'] > 0, df['INVESTIGATI'] / df['ENTRATI'], 0.0)

        return df

    def _apply_filters(self, df: pd.DataFrame, filters: Dict[str, Any]) -> pd.DataFrame:
        filtered = df.copy()

        #apply every filter by column
        for key, value in filters.items():
            if value is not None and key in filtered.columns:
                if isinstance(value, list):
                    filtered = filtered[filtered[key].isin(value)]
                else:
                    filtered = filtered[filtered[key] == value]

        return filtered

    def run(self, filters: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:

        #loading datasets
        raw_allarmi = pd.read_csv(self.allarmi_path)
        raw_tipologia = pd.read_csv(self.tipologia_path)

        #cleaning
        df_allarmi = self._clean_allarmi(raw_allarmi)
        df_tipologia = self._clean_tipologia(raw_tipologia)

        #applying filters
        if filters:
            df_allarmi_f = self._apply_filters(df_allarmi, filters)
            df_tipologia_f = self._apply_filters(df_tipologia, filters)
        else:
            df_allarmi_f = df_allarmi.copy()
            df_tipologia_f = df_tipologia.copy()

        #summary of what data the rest of the pipeline will work on
        if filters:
            print(f"DataAgent — filters applied: {filters}")
        else:
            print("DataAgent — no filters applied (full dataset)")
        print(f"   ALLARMI:    {len(df_allarmi_f):>5} rows  (of {len(df_allarmi)} total)")
        print(f"   TIPOLOGIA:  {len(df_tipologia_f):>5} rows  (of {len(df_tipologia)} total)")

        return {
            'allarmi_full': df_allarmi,
            'tipologia_full': df_tipologia,
            'allarmi': df_allarmi_f,
            'tipologia': df_tipologia_f,
            'filters_applied': filters, 
        }

In [19]:
#data agent esecution
data_agent = DataAgent('data/raw/ALLARMI.csv', 'data/raw/TIPOLOGIA_VIAGGIATORE.csv')
data_output = data_agent.run(filters=None)

display(Markdown("df1"))
display(data_output['allarmi'].head(5))
display(Markdown("df2"))
display(data_output['tipologia'].head(5))

DataAgent — no filters applied (full dataset)
   ALLARMI:     5080 rows  (of 5080 total)
   TIPOLOGIA:   5095 rows  (of 5095 total)


df1

,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,...,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli,ROTTA
0,Voli con Allarmi,FCO,IST,2024,1,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,...,1,Manuale,NaN,N/A,Turchia,ITA,5,Italia,1,IST → FCO
1,Viaggiatori con Allarmi,CIA,STN,2024,2,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,...,5,Manuale,NaN,N/A,Regno Unito,ITA,5,Italia,5,STN → CIA
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,1,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,...,110,TSC,NaN,N/A,Regno Unito,ITA,5,Italia,110,LHR → FCO
3,Voli con Allarmi,MXP,LHR,2024,2,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,...,1,SDI,NaN,N/A,Regno Unito,ITA,2,Italia,1,LHR → MXP
4,Viaggiatori con Allarmi,PSA,BRS,2024,2,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,...,2,INTERPOL,NaN,N/A,Regno Unito,ITA,8,Italia,2,BRS → PSA


df2

,NAZIONALITA,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,GIORNO_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,...,note_operatore,codice_rischio,Tipo Documento,FASCIA ETA,3nazionalita,compagnia%aerea,num volo,ROTTA,TASSO_ALLARME,TASSO_INVESTIGAZIONE
0,ALB,NAP,DUR,2024,2,13,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,...,NaN,NaN,Passaporto,N.D.,ALB,Fly Dubai,FZ1681,DUR → NAP,0.000000,1.0
1,NaN,FCO,JFK,2024,1,22,2024-01-22 16:35:00,Fiumicino,John F Kennedy International,Roma,...,NaN,NaN,Carta d'identità,18-30,ALB,ITA Airways,AZ0609,JFK → FCO,1.000000,0.0
2,ALB,TSF,TIA,2024,2,4,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,...,NaN,NaN,N.D.,31-45,ALB,Ryanair DAC,FR8400,TIA → TSF,0.224138,1.0
3,AFG,FCO,IST,2024,1,25,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,...,NaN,NaN,N.D.,61+,AFG,Turkish Airlines,TK1865,IST → FCO,0.000000,1.0
4,ALB,BGY,MLE,2024,2,13,NaT,Orio al Serio,Male International,Bergamo,...,NaN,NaN,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571,MLE → BGY,0.500000,1.0


---

## 2 — Baseline Agent

The Baseline Agent is in charge of building historical baselines per route, nationality and document type, using rolling averages and seasonal decomposition.

In [20]:
class BaselineAgent:

    def __init__(self):
        self.baselines = {}
        self.feature_matrix = None

    def _compute_group_stats(self, df, group_col, value_col, label):
        #aggregate sum, mean, std, count for a given grouping column
        grouped = df.groupby(group_col)[value_col].agg(['sum','mean','std','count']).reset_index()
        grouped.columns = [group_col, f'{label}_sum', f'{label}_mean',
                           f'{label}_std', f'{label}_count']
        grouped[f'{label}_std'] = grouped[f'{label}_std'].fillna(0)
        return grouped

    def _compute_rolling_baseline(self, df, group_col, date_col, value_col, window=3):
        #compute monthly rolling average and std per group
        df_ts = df.copy()
        df_ts['YEAR_MONTH'] = df_ts[date_col].dt.to_period('M')
        monthly = df_ts.groupby([group_col, 'YEAR_MONTH'])[value_col].sum().reset_index()
        monthly = monthly.sort_values([group_col, 'YEAR_MONTH'])
        monthly[f'{value_col}_rolling_avg'] = monthly.groupby(group_col)[value_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean())
        monthly[f'{value_col}_rolling_std'] = monthly.groupby(group_col)[value_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).std().fillna(0))
        return monthly

    def _seasonal_decomposition(self, df, group_col, value_col):
        #compute mean and std per group per month to capture seasonal patterns
        seasonal = df.groupby([group_col, 'MESE_PARTENZA'])[value_col].agg(['mean','std']).reset_index()
        seasonal.columns = [group_col, 'MESE_PARTENZA',
                            f'{value_col}_seasonal_mean', f'{value_col}_seasonal_std']
        seasonal[f'{value_col}_seasonal_std'] = seasonal[f'{value_col}_seasonal_std'].fillna(0)
        return seasonal

    def _build_feature_matrix(self, df_allarmi, df_tipologia):
        #route-level alarm statistics
        allarmi_route = self._compute_group_stats(df_allarmi, 'ROTTA', 'TOT', 'allarmi_tot')

        #pivot alarm occurrences and reasons by route
        occ_pivot = df_allarmi.groupby(['ROTTA','OCCORRENZE']).size().unstack(fill_value=0)
        occ_pivot.columns = ['occ_' + c.replace(' ','_')[:25] for c in occ_pivot.columns]
        occ_pivot = occ_pivot.reset_index()

        motivo_pivot = df_allarmi.groupby(['ROTTA','MOTIVO_ALLARME']).size().unstack(fill_value=0)
        motivo_pivot.columns = ['motivo_' + str(c) for c in motivo_pivot.columns]
        motivo_pivot = motivo_pivot.reset_index()

        #route-level passenger statistics
        tip_route = df_tipologia.groupby('ROTTA').agg(
            entrati_sum=('ENTRATI','sum'), investigati_sum=('INVESTIGATI','sum'),
            allarmati_sum=('ALLARMATI','sum'), entrati_mean=('ENTRATI','mean'),
            allarmati_mean=('ALLARMATI','mean'), tasso_allarme_mean=('TASSO_ALLARME','mean'),
            tasso_invest_mean=('TASSO_INVESTIGAZIONE','mean'), n_records=('ENTRATI','count'),
            n_nazionalita=('NAZIONALITA','nunique'), n_doc_types=('TIPO_DOCUMENTO','nunique'),
        ).reset_index()

        #pivot control outcomes by route
        esito_pivot = df_tipologia.groupby(['ROTTA','ESITO_CONTROLLO']).size().unstack(fill_value=0)
        esito_pivot.columns = ['esito_' + str(c).replace(' ','_') for c in esito_pivot.columns]
        esito_pivot = esito_pivot.reset_index()

        #merge all features and compute derived rates
        features = allarmi_route.merge(occ_pivot, on='ROTTA', how='outer')
        features = features.merge(motivo_pivot, on='ROTTA', how='outer')
        features = features.merge(tip_route, on='ROTTA', how='outer')
        features = features.merge(esito_pivot, on='ROTTA', how='outer').fillna(0)
        features['alert_rate'] = np.where(features['entrati_sum']>0,
            features['allarmati_sum']/features['entrati_sum'], 0.0)
        features['investigation_rate'] = np.where(features['entrati_sum']>0,
            features['investigati_sum']/features['entrati_sum'], 0.0)
        return features

    def run(self, data: Dict[str, Any]) -> Dict[str, Any]:
        df_a, df_t = data['allarmi'], data['tipologia']

        #baselines per gate, route, nationality, document type
        self.baselines['gate'] = self._compute_group_stats(df_a, 'AREOPORTO_ARRIVO', 'TOT', 'gate')
        self.baselines['route'] = self._compute_group_stats(df_a, 'ROTTA', 'TOT', 'route')
        self.baselines['nationality'] = self._compute_group_stats(df_t, 'NAZIONALITA', 'ALLARMATI', 'nat')
        self.baselines['document'] = self._compute_group_stats(df_t, 'TIPO_DOCUMENTO', 'ALLARMATI', 'doc')

        #rolling averages (only if date column is populated)
        valid_dates = df_a.dropna(subset=['DATA_PARTENZA'])
        if len(valid_dates) > 0:
            self.baselines['rolling_route'] = self._compute_rolling_baseline(
                valid_dates, 'ROTTA', 'DATA_PARTENZA', 'TOT', window=3)
        else:
            self.baselines['rolling_route'] = pd.DataFrame()

        #seasonal decomposition by month
        self.baselines['seasonal_gate'] = self._seasonal_decomposition(df_a, 'AREOPORTO_ARRIVO', 'TOT')

        #feature matrix aggregated at route level
        self.feature_matrix = self._build_feature_matrix(df_a, df_t)

        #global summary statistics used as anomaly detection thresholds
        global_stats = {
            'alert_rate_mean': round(float(self.feature_matrix['alert_rate'].mean()), 4),
            'alert_rate_std': round(float(self.feature_matrix['alert_rate'].std()), 4),
            'alert_rate_p75': round(float(self.feature_matrix['alert_rate'].quantile(0.75)), 4),
            'alert_rate_p95': round(float(self.feature_matrix['alert_rate'].quantile(0.95)), 4),
            'investigation_rate_mean': round(float(self.feature_matrix['investigation_rate'].mean()), 4),
            'investigation_rate_std': round(float(self.feature_matrix['investigation_rate'].std()), 4),
            'allarmi_tot_mean_global': round(float(self.feature_matrix['allarmi_tot_sum'].mean()), 1),
            'allarmi_tot_std_global': round(float(self.feature_matrix['allarmi_tot_sum'].std()), 1),
        }

        return {
            'baselines': self.baselines,
            'feature_matrix': self.feature_matrix,
            'global_stats': global_stats,
        }

In [21]:
#baseline agent esecution
baseline_agent = BaselineAgent()
baseline_output = baseline_agent.run(data_output)

display(baseline_output['feature_matrix'].head(5))

,ROTTA,allarmi_tot_sum,allarmi_tot_mean,allarmi_tot_std,allarmi_tot_count,occ_???,occ_ALLARMATI,occ_Allarmi_Chiusi,occ_Allarmi_Chiusi_con_Azione,occ_Allarmi_NON_Chiusi,...,n_nazionalita,n_doc_types,esito_FERMATO,esito_IN_ATTESA,esito_N.D.,esito_OK,esito_RESPINTO,esito_SEGNALATO,alert_rate,investigation_rate
0,ABJ → CAG,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000,1.0
1,ADB → BGY,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.000000,1.0
2,ADB → FCO,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.000000,1.0
3,ADB → MXP,181.0,36.2,75.373735,5.0,0.0,0.0,0.0,0.0,0.0,...,1.0,3.0,1.0,2.0,0.0,1.0,1.0,0.0,0.571429,1.0
4,ADB → NAP,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,1.0


---

## 3 — Outlier Detection Agent

The Outlier Detection Agent is in charge of applying IsolationForest, LOF and Z-score. The consensus voting is 2 out of 3. The agent analyzes every result and explains why every method flagged those anomalies

In [22]:
class OutlierDetectionAgent:

    def __init__(self, contamination=0.1, z_threshold=2.5):
        self.contamination = contamination
        self.z_threshold = z_threshold

    def _select_numeric_features(self, df):
        #select and scale all numeric columns except the route identifier
        numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'ROTTA']
        X = df[numeric_cols].fillna(0).values
        scaler = StandardScaler()
        return scaler.fit_transform(X), numeric_cols, scaler

    def _isolation_forest(self, X):
        #tree-based method: isolates anomalies by random partitioning
        clf = IsolationForest(contamination=self.contamination, random_state=42,
                              n_estimators=200, n_jobs=-1)
        return clf.fit_predict(X), clf.decision_function(X)

    def _local_outlier_factor(self, X):
        #density-based method: flags points in low-density neighborhoods
        n = min(20, max(2, len(X) - 1))
        clf = LocalOutlierFactor(n_neighbors=n, contamination=self.contamination, novelty=False)
        return clf.fit_predict(X), clf.negative_outlier_factor_

    def _zscore_detection(self, X, feature_names):
        #flags routes where at least one feature exceeds the z-score threshold
        z = np.abs(stats.zscore(X, nan_policy='omit'))
        labels = np.where((z > self.z_threshold).any(axis=1), -1, 1)
        return labels, z.max(axis=1), [feature_names[i] for i in z.argmax(axis=1)]

    def run(self, baseline_data):
        fm = baseline_data['feature_matrix'].copy()
        baselines = baseline_data['baselines']

        if len(fm) < 5:
            fm['is_anomaly'] = False
            return {'feature_matrix': fm, 'anomalies': fm[fm['is_anomaly']],
                    'feature_names': []}

        #enrich feature matrix with rolling baseline deviation
        rolling = baselines.get('rolling_route', pd.DataFrame())
        if len(rolling) > 0:
            #take the last rolling average per route (most recent window)
            latest_rolling = (rolling.sort_values('YEAR_MONTH')
                                    .groupby('ROTTA')[['TOT_rolling_avg', 'TOT_rolling_std']]
                                    .last()
                                    .reset_index())
            fm = fm.merge(latest_rolling, on='ROTTA', how='left')
            fm['rolling_deviation'] = np.where(
                fm['TOT_rolling_std'] > 0,
                (fm['allarmi_tot_sum'] - fm['TOT_rolling_avg']) / fm['TOT_rolling_std'],
                0.0)
            fm.drop(columns=['TOT_rolling_avg', 'TOT_rolling_std'], inplace=True)
        else:
            fm['rolling_deviation'] = 0.0

        #enrich feature matrix with seasonal baseline deviation
        seasonal = baselines.get('seasonal_gate', pd.DataFrame())
        if len(seasonal) > 0:
            #aggregate seasonal mean across all months per route (approximation at route level)
            seasonal_agg = (seasonal.groupby('AREOPORTO_ARRIVO')
                                    ['TOT_seasonal_mean'].mean()
                                    .reset_index()
                                    .rename(columns={'AREOPORTO_ARRIVO': 'gate',
                                                    'TOT_seasonal_mean': 'gate_seasonal_mean'}))
            #extract arrival gate from ROTTA and join
            fm['gate'] = fm['ROTTA'].str.split(' → ').str[1]
            fm = fm.merge(seasonal_agg, on='gate', how='left')
            fm['seasonal_deviation'] = np.where(
                fm['gate_seasonal_mean'] > 0,
                fm['allarmi_tot_sum'] / fm['gate_seasonal_mean'],
                1.0)
            fm.drop(columns=['gate', 'gate_seasonal_mean'], inplace=True)
        else:
            fm['seasonal_deviation'] = 1.0

        fm[['rolling_deviation', 'seasonal_deviation']] = (
            fm[['rolling_deviation', 'seasonal_deviation']].fillna(0))

        #scale features for model input (now includes rolling_deviation and seasonal_deviation)
        X, feat_names, scaler = self._select_numeric_features(fm)

        #run the three detection methods independently
        fm['IF_label'], fm['IF_score'] = self._isolation_forest(X)
        fm['LOF_label'], fm['LOF_score'] = self._local_outlier_factor(X)
        fm['Zscore_label'], fm['Zscore_max'], fm['Zscore_top_feature'] = self._zscore_detection(X, feat_names)

        #a route is anomalous if at least 2 out of 3 methods agree
        fm['anomaly_votes'] = ((fm['IF_label'] == -1).astype(int) +
                            (fm['LOF_label'] == -1).astype(int) +
                            (fm['Zscore_label'] == -1).astype(int))
        fm['is_anomaly'] = fm['anomaly_votes'] >= 2

        #weighted severity score: IF 40%, LOF 30%, Z-score 30%
        fm['severity_score'] = (
            (1 - (fm['IF_score'] - fm['IF_score'].min()) / (fm['IF_score'].max() - fm['IF_score'].min() + 1e-10)) * 0.4 +
            (1 - (fm['LOF_score'] - fm['LOF_score'].min()) / (fm['LOF_score'].max() - fm['LOF_score'].min() + 1e-10)) * 0.3 +
            (fm['Zscore_max'] / (fm['Zscore_max'].max() + 1e-10)) * 0.3)

        anomalies = fm[fm['is_anomaly']].sort_values('severity_score', ascending=False)

        return {
            'feature_matrix': fm,
            'anomalies': anomalies,
            'feature_names': feat_names,
        }

In [23]:
#outlier detection agent execution
outlier_agent = OutlierDetectionAgent(contamination=0.1, z_threshold=2.5)
outlier_output = outlier_agent.run(baseline_output)

# Display consensus anomalies (flagged by at least 2 out of 3 methods)
cols = ['ROTTA','alert_rate','allarmati_sum','entrati_sum','IF_label','LOF_label',
        'Zscore_label','anomaly_votes','severity_score','Zscore_top_feature']
dc = [c for c in cols if c in outlier_output['anomalies'].columns]
display(outlier_output['anomalies'][dc].head(15))

,ROTTA,alert_rate,allarmati_sum,entrati_sum,IF_label,LOF_label,Zscore_label,anomaly_votes,severity_score,Zscore_top_feature
583,STN → BGY,0.003547,36.0,10150.0,-1,1,-1,2,0.657999,occ_Nulla_a_procedere_INT
334,LGW → MXP,0.241667,29.0,120.0,-1,1,-1,2,0.636200,seasonal_deviation
663,TIA → BGY,0.188656,4540.0,24065.0,-1,1,-1,2,0.614166,occ_Mancato_aggiornamento_Sch
664,TIA → BLQ,0.131683,3872.0,29404.0,-1,1,-1,2,0.590003,occ_Respinto/a
260,IST → FCO,0.352941,12.0,34.0,-1,1,-1,2,0.526795,occ_N/C
138,DOH → MXP,1.200000,6.0,5.0,-1,1,-1,2,0.512238,occ_???
688,TIA → TSF,0.187229,2202.0,11761.0,-1,1,-1,2,0.493646,occ_Inammissibilita_Schengen_
675,TIA → MXP,0.174451,3115.0,17856.0,-1,1,-1,2,0.482294,occ_Allarmi_Chiusi_con_Azione
691,TIA → VRN,0.681586,6944.0,10188.0,-1,1,-1,2,0.479672,allarmati_sum
534,SCL → MXP,0.000000,0.0,-500.0,1,-1,-1,2,0.459672,entrati_mean


---

## 4 — Risk Profiling Agent
The Risk Profiling Agent is applying business rules (e.g., 3x baseline → HIGH, REJECTED/HALTED, severity, 3/3 consensus). The agent validates classifications and may suggest upgrades or downgrades

In [24]:
class RiskProfilingAgent:

    def __init__(self):
        pass

    def _classify(self, anomalies, gs):
        bar = gs['alert_rate_mean']
        tmean = gs['allarmi_tot_mean_global']
        tstd = gs['allarmi_tot_std_global']
        levels, reasons_list = [], []

        #explicit escalation: HIGH > MEDIUM > LOW
        def escalate(current, target):
            order = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
            return target if order[target] > order[current] else current

        for _, r in anomalies.iterrows():
            reasons, lv = [], 'LOW'

            #alert rate vs baseline thresholds
            ar = r.get('alert_rate', 0)
            if bar > 0 and ar > 3 * bar:
                lv = escalate(lv, 'HIGH')
                reasons.append(f"Alert rate {ar/bar:.1f}x baseline")
            elif bar > 0 and ar > 2 * bar:
                lv = escalate(lv, 'MEDIUM')
                reasons.append(f"Alert rate {ar/bar:.1f}x baseline")

            #alarm volume above mean + 2std
            thr = tmean + 2 * tstd
            if r.get('allarmi_tot_sum', 0) > thr:
                lv = escalate(lv, 'HIGH')
                reasons.append(f"Volume {r['allarmi_tot_sum']:.0f}>{thr:.0f}")

            #presence of rejected or stopped passengers
            rej = r.get('esito_RESPINTO', 0) + r.get('esito_FERMATO', 0)
            if rej > 0:
                lv = escalate(lv, 'MEDIUM')
                reasons.append(f"Rejected/Stopped: {rej:.0f}")

            #high composite severity score
            if r.get('severity_score', 0) > 0.8:
                lv = escalate(lv, 'HIGH')
                reasons.append(f"Severity {r['severity_score']:.2f}")

            #all three detection methods agree
            if r.get('anomaly_votes', 0) == 3:
                lv = escalate(lv, 'HIGH')
                reasons.append("All 3 methods agree")

            if not reasons:
                reasons.append("Statistical anomaly (>=2 methods)")

            levels.append(lv)
            reasons_list.append(' | '.join(reasons))

        anomalies = anomalies.copy()
        anomalies['risk_level'] = levels
        anomalies['risk_reasons'] = reasons_list
        return anomalies

    def run(self, outlier_data, baseline_data):
        anomalies = outlier_data['anomalies'].copy()
        gs = baseline_data['global_stats']

        if len(anomalies) == 0:
            return {'risk_profiles': pd.DataFrame(), 'summary': {}, 'global_stats': gs}

        #classify each anomalous route and sort by risk level
        classified = self._classify(anomalies, gs)
        classified = classified.sort_values('risk_level',
            key=lambda x: x.map({'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}))

        #risk level distribution
        summary = classified['risk_level'].value_counts().to_dict()

        return {
            'risk_profiles': classified,
            'summary': summary,
            'global_stats': gs,
        }

In [25]:
#risk profiling agent execution
risk_agent = RiskProfilingAgent()
risk_output = risk_agent.run(outlier_output, baseline_output)

#display risk profiles sorted by level
rc = ['ROTTA','risk_level','alert_rate','allarmati_sum','entrati_sum',
      'severity_score','anomaly_votes','risk_reasons']
display(risk_output['risk_profiles'][[c for c in rc if c in risk_output['risk_profiles'].columns]].head(15))

,ROTTA,risk_level,alert_rate,allarmati_sum,entrati_sum,severity_score,anomaly_votes,risk_reasons
57,BHX → FCO,HIGH,0.000000,0.0,1.0,0.429353,3,Rejected/Stopped: 1 | All 3 methods agree
684,TIA → RMI,HIGH,0.135599,395.0,2913.0,0.309639,3,Rejected/Stopped: 8 | All 3 methods agree
482,PVG → MXP,HIGH,1.200000,6.0,5.0,0.240504,2,Alert rate 5.8x baseline
348,LHR → LIN,HIGH,0.058824,1.0,17.0,0.417466,2,Volume 13288>10201 | Rejected/Stopped: 6
481,PVG → FCO,HIGH,0.750000,6.0,8.0,0.164260,2,Alert rate 3.6x baseline | Rejected/Stopped: 2
53,BFS → BGY,HIGH,0.700000,7.0,10.0,0.162393,2,Alert rate 3.4x baseline | Rejected/Stopped: 3
183,EVN → VCE,HIGH,2.000000,2.0,1.0,0.226015,2,Alert rate 9.6x baseline | Rejected/Stopped: 1
266,IST → VCE,HIGH,0.782609,18.0,23.0,0.236881,2,Alert rate 3.8x baseline | Rejected/Stopped: 4
488,RAK → CIA,HIGH,1.000000,4.0,4.0,0.275373,2,Alert rate 4.8x baseline | Rejected/Stopped: 3
262,IST → NAP,HIGH,0.666667,2.0,3.0,0.172946,2,Alert rate 3.2x baseline


---

## 5 — Report Agent

The Report Agent is in charge of writing a final report

In [26]:
class ReportAgent:

    def __init__(self, llm_instance):
        self.llm = llm_instance

    def _build_summary(self, rp, max_items=15):
        #format anomalies grouped by risk level for the LLM prompt
        lines = []
        for lv in ['HIGH', 'MEDIUM', 'LOW']:
            sub = rp[rp['risk_level'] == lv].head(max_items)
            if len(sub) == 0:
                continue
            lines.append(f"\n=== {lv} RISK ({len(sub)} anomalies) ===")
            for _, r in sub.iterrows():
                #detection method consensus: which of the 3 methods agreed
                methods = []
                if r.get('IF_label', 1) == -1:
                    methods.append('IsolationForest')
                if r.get('LOF_label', 1) == -1:
                    methods.append('LOF')
                if r.get('Zscore_label', 1) == -1:
                    methods.append('Z-score')
                methods_str = ', '.join(methods) if methods else 'N/A'

                lines.append(
                    f"  Route: {r.get('ROTTA','N/A')}\n"
                    f"    Alert rate: {r.get('alert_rate',0):.4f} | "
                    f"Alarmed: {r.get('allarmati_sum',0):.0f} | "
                    f"Entered: {r.get('entrati_sum',0):.0f}\n"
                    f"    Severity: {r.get('severity_score',0):.3f} | "
                    f"Votes: {r.get('anomaly_votes',0)}/3 | "
                    f"Methods: {methods_str}\n"
                    f"    Top Z-score feature: {r.get('Zscore_top_feature','N/A')}\n"
                    f"    Reason: {r.get('risk_reasons','N/A')}")
        return '\n'.join(lines)

    def run(self, risk_data, filters_applied=None):
        rp = risk_data['risk_profiles']
        rs = risk_data['summary']
        gs = risk_data['global_stats']

        #edge case: no anomalies detected
        if len(rp) == 0:
            msg = self.llm.invoke(
                "Write a brief professional report stating no anomalies were detected "
                "in airport transit data. Max 50 words.").content
            return {'report': msg}

        #context: what subset of data was analysed
        if filters_applied:
            scope = f"Analysis scope: filtered dataset — {filters_applied}"
        else:
            scope = "Analysis scope: full dataset, no filters applied."

        #build structured anomaly summary and inject into prompt
        summary = self._build_summary(rp)
        prompt = (
            f"You are a senior border-control analyst. Write a professional "
            f"Transit Anomaly Report.\n\n"
            f"ANALYSIS SCOPE:\n{scope}\n\n"
            f"GLOBAL BASELINES:\n"
            f"- Avg alert rate: {gs.get('alert_rate_mean',0):.4f}\n"
            f"- Std: {gs.get('alert_rate_std',0):.4f}\n"
            f"- P95: {gs.get('alert_rate_p95',0):.4f}\n"
            f"- Avg alarms/route: {gs.get('allarmi_tot_mean_global',0):.1f}\n\n"
            f"RISK DISTRIBUTION: HIGH={rs.get('HIGH',0)}, MEDIUM={rs.get('MEDIUM',0)}, "
            f"LOW={rs.get('LOW',0)}\n\n"
            f"ANOMALIES:\n{summary}\n\n"
            f"STRUCTURE:\n"
            f"1. Executive Summary (2-3 sentences, mention the analysis scope)\n"
            f"2. Each HIGH risk: description, which detection methods flagged it, "
            f"the top anomalous feature, deviation from baseline, recommended action\n"
            f"3. MEDIUM and LOW as groups\n"
            f"4. Overall Risk Assessment\n"
            f"Professional, concise."
        )

        #generate report via LLM
        report = self.llm.invoke(prompt).content

        return {'report': report}

In [27]:
#report agent execution
report_agent = ReportAgent(llm_instance=llm)
report_output = report_agent.run(risk_output)

display(Markdown(report_output['report']))

Okay, here's a Transit Anomaly Report based on the provided data and your instructions.

---

**TRANSIT ANOMALY REPORT – DATE: October 26, 2023**

**1. Executive Summary:**

This report analyzes a full dataset of transit routes to identify anomalies exceeding established baselines.  The analysis revealed 15 high-risk routes exhibiting significant deviations from expected alert rates, primarily driven by unusually high volumes of alerts or specific feature patterns.  A further 15 medium-risk routes showed notable anomalies, while 15 low-risk routes displayed statistical anomalies requiring further investigation.  Recommendations focus on enhanced monitoring and potential process adjustments for the high-risk routes to mitigate potential security risks.

**2. HIGH RISK (15 Anomalies)**

| Route          | Alert Rate | Alarmed | Entered | Severity | Votes | Methods           | Top Z-Score Feature             | Reason                                  | Recommended Action                   

Okay, here's a Transit Anomaly Report based on the provided data and your instructions.

---

**TRANSIT ANOMALY REPORT – DATE: October 26, 2023**

**1. Executive Summary:**

This report analyzes a full dataset of transit routes to identify anomalies exceeding established baselines.  The analysis revealed 15 high-risk routes exhibiting significant deviations from expected alert rates, primarily driven by unusually high volumes of alerts or specific feature patterns.  A further 15 medium-risk routes showed notable anomalies, while 15 low-risk routes displayed statistical anomalies requiring further investigation.  Recommendations focus on enhanced monitoring and potential process adjustments for the high-risk routes to mitigate potential security risks.

**2. HIGH RISK (15 Anomalies)**

| Route          | Alert Rate | Alarmed | Entered | Severity | Votes | Methods           | Top Z-Score Feature             | Reason                                  | Recommended Action                                    |
|----------------|------------|---------|--------|----------|-------|--------------------|----------------------------------|-------------------------------------------|------------------------------------------------------|
| BHX → FCO      | 0.0000     | 0       | 1      | 0.429    | 3/3   | IsolationForest, LOF, Z-score | investigation_rate             | Rejected/Stopped: 1 (All methods agree)     | Prioritize review of passenger manifests for this route. |
| TIA → RMI      | 0.1356     | 395     | 2913   | 0.310    | 3/3   | IsolationForest, LOF, Z-score | occ_Allarmi_generati            | Rejected/Stopped: 8 (All methods agree)   | Investigate potential data inconsistencies.           |
| PVG → MXP      | 1.2000     | 6       | 5      | 0.241    | 2/3   | IsolationForest, Z-score | occ_Allarmi_generati_da_INTER  | Alert rate 5.8x baseline              | Review INTER-related data sources and processes.     |
| LHR → LIN      | 0.0588     | 1       | 17     | 0.417    | 2/3   | IsolationForest, Z-score | occ_Viaggiatori_entrati_nel_S   | Volume 13288 > 10201 (Rejected/Stopped: 6) | Investigate potential volume anomalies.              |
| PVG → FCO      | 0.7500     | 6       | 8      | 0.164    | 2/3   | IsolationForest, Z-score | rolling_deviation             | Alert rate 3.6x baseline (Rejected/Stopped: 2)| Examine rolling average data for anomalies.        |
| BFS → BGY      | 0.7000     | 7       | 10     | 0.162    | 2/3   | LOF, Z-score | occ_Allarmi_generati_da_INTER  | Alert rate 3.4x baseline (Rejected/Stopped: 3)| Investigate potential data inconsistencies.           |
| EVN → VCE      | 2.0000     | 2       | 1      | 0.226    | 2/3   | LOF, Z-score | occ_Nulla_a_procedere_NSIS    | Alert rate 9.6x baseline (Rejected/Stopped: 1)| Review NSIS data and processing rules.             |
| IST → VCE      | 0.7826     | 18      | 23     | 0.237    | 2/3   | IsolationForest, Z-score | occ_Viaggiatori_con_Allarmi     | Alert rate 3.8x baseline (Rejected/Stopped: 4)| Investigate passenger travel patterns.              |
| RAK → CIA      | 1.0000     | 4       | 4      | 0.275    | 2/3   | LOF, Z-score | occ_Errata_segnalazione_NSIS  | Alert rate 4.8x baseline (Rejected/Stopped: 3)| Validate NSIS data accuracy and reporting.          |
| IST → NAP      | 0.6667     | 2       | 3      | 0.173    | 2/3   | IsolationForest, Z-score | occ_Allarmi_generati_da_SDI/N   | Alert rate 3.2x baseline (Rejected/Stopped: 2)| Investigate SDI/N data discrepancies.             |
| Lcy → Flr      | 0.0000     | 0       | 0      | 0.453    | 2/3   | LOF, Z-score | allarmi_tot_mean             | Volume 100000 > 10201 (Rejected/Stopped: 3)| Investigate potential data volume issues.          |
| TIA → VRN      | 0.6816     | 6944    | 10188  | 0.480    | 2/3   | IsolationForest, Z-score | allarmati_sum               | Alert rate 3.3x baseline (Rejected/Stopped: 3)| Review VRN data processing and validation.         |
| GRU → MXP      | 1.0000     | 2       | 2      | 0.290    | 2/3   | LOF, Z-score | occ_Altro                  | Alert rate 4.8x baseline (Rejected/Stopped: 1)| Investigate the "Altro" category for anomalies.   |
| CMN → BLQ      | 0.6667     | 2       | 3      | 0.264    | 2/3   | IsolationForest, Z-score | occ_Notifica_Atti/Provv       | Alert rate 3.2x baseline (Rejected/Stopped: 2)| Review Attestation/Provisioning data.            |
| DOH → MXP      | 1.2000     | 6       | 5      | 0.512    | 2/3   | IsolationForest, Z-score | occ_???                    | Alert rate 5.8x baseline (Rejected/Stopped: 1)| Investigate the unknown "occ_???" feature.       |

**3. MEDIUM RISK (15 Anomalies)**

(Detailed list of Medium Risk routes and their analysis would be included here – similar format to High Risk, but with less emphasis on immediate action).

**4. LOW RISK (15 Anomalies)**

(Detailed list of Low Risk routes and their analysis would be included here – focusing on statistical anomalies and potential data quality issues).

**5. Overall Risk Assessment:**

The analysis indicates a significant number of transit routes exhibiting anomalous behavior, primarily driven by high alert rates and specific feature patterns. The High-Risk group represents a critical area of concern, requiring immediate attention to investigate potential security vulnerabilities and data integrity issues. Medium-Risk routes warrant further investigation and process review, while Low-Risk anomalies should be monitored for potential trends.  Continuous monitoring and refinement of anomaly detection methods are recommended to improve overall transit security.

---

**Note:**  This report is based on the limited data provided. A full investigation would require access to more detailed data sources and a deeper understanding of the underlying transit system.  The "Recommended Action" section is intended as a starting point and should be adapted based on further analysis.  The "Votes" column indicates the consensus among the anomaly detection methods used (Isolation Forest, LOF, and Z-score).  If a method disagrees significantly with the others, it suggests a potential data quality issue or a more complex anomaly that requires manual review.

---

## 6 — Orchestrator

The orchestrator runs the full multi-agent anomaly detection pipeline

In [28]:
def run_anomaly_detection_pipeline(
    allarmi_path='data/raw/ALLARMI.csv', tipologia_path='data/raw/TIPOLOGIA_VIAGGIATORE.csv',
    filters=None, contamination=0.1, z_threshold=2.5,
):

    a1 = DataAgent(allarmi_path, tipologia_path)
    data = a1.run(filters=filters)

    a2 = BaselineAgent()
    baselines = a2.run(data)

    a3 = OutlierDetectionAgent(contamination=contamination, z_threshold=z_threshold)
    outliers = a3.run(baselines)

    a4 = RiskProfilingAgent()
    risks = a4.run(outliers, baselines)

    a5 = ReportAgent(llm_instance=llm)
    report = a5.run(risks, filters_applied=data.get('filters_applied'))

    return {'data': data, 'baselines': baselines, 'outliers': outliers,
            'risks': risks, 'report': report}

We defined a function that translates a natural language query into a filters dictionary compatible with run_anomaly_detection_pipeline()

In [29]:
def _apply_filters(self, df: pd.DataFrame, filters: Dict[str, Any]) -> pd.DataFrame:

    # NEW: Text-to-Pandas mode
    if "__pandas_expr__" in filters:
        expr = filters["__pandas_expr__"]
        try:
            result = df.query(expr)
            if len(result) == 0:
                print(f"WARNING: pandas expr '{expr}' returned 0 rows → using full dataset.")
                return df.copy()
            return result
        except Exception as e:
            print(f"WARNING: pandas expr failed ({e}) → using full dataset.")
            return df.copy()

    # ORIGINAL: dict-based filter mode (backward compatible)
    filtered = df.copy()
    for key, value in filters.items():
        if key.startswith("__"):
            continue  # skip metadata keys
        if value is not None and key in filtered.columns:
            if isinstance(value, list):
                filtered = filtered[filtered[key].isin(value)]
            else:
                filtered = filtered[filtered[key] == value]
    return filtered

---

---
## 7 — Natural Language Query (Prototype, not fully working yet)

Write your request in Italian (or English) in the `USER_QUERY` variable.  
The LLM automatically translates it into filters and runs the pipeline on the correct subset.

> **Examples of valid queries:**
> - `"analyze flights arriving at Malpensa in March"`
> - `"I want to see only flights from Istanbul"`
> - `"focus on Albania in 2024"`
> - `"all data without filters"`

In [30]:
def parse_query_to_filters(user_query: str, verbose: bool = True) -> Optional[Dict[str, Any]]:
    """
    Converts a natural language query into a pandas boolean expression
    applied directly on df_allarmi and df_tipologia after cleaning.
    Returns a dict with key 'pandas_expr' (string) or None for full dataset.
    """

    # Schema info injected into the prompt so the LLM knows what columns exist
    ALLARMI_SCHEMA = """
ALLARMI dataframe columns (after cleaning):
- AREOPORTO_ARRIVO   : str  — IATA arrival airport code (e.g. 'FCO', 'MXP', 'BGY', 'VCE', 'BLQ', 'TSF')
- AREOPORTO_PARTENZA : str  — IATA departure airport code (e.g. 'IST', 'TIA', 'DXB', 'SAW')
- MESE_PARTENZA      : int  — departure month (1=Jan ... 12=Dec)
- ANNO_PARTENZA      : int  — departure year (e.g. 2024)
- PAESE_PART         : str  — departure country UPPERCASE (e.g. 'ALBANIA', 'TURCHIA', 'MAROCCO')
- PAESE_ARR          : str  — arrival country UPPERCASE (e.g. 'ITALIA')
- ZONA               : int  — geographic zone number
- TOT                : int  — total flights on the route
- ROTTA              : str  — route string e.g. 'IST → FCO'
- MOTIVO_ALLARME     : str  — alarm reason
"""

    TIPOLOGIA_SCHEMA = """
TIPOLOGIA dataframe columns (after cleaning):
- AREOPORTO_ARRIVO   : str  — same as above
- AREOPORTO_PARTENZA : str  — same as above
- MESE_PARTENZA      : int  — same as above
- ANNO_PARTENZA      : int  — same as above
- PAESE_PART         : str  — same as above
- NAZIONALITA        : str  — passenger nationality UPPERCASE (e.g. 'ALBANESE', 'TURCO')
- ALLARMATI          : int  — number of alarmed passengers
- INVESTIGATI        : int  — number of investigated passengers
- ENTRATI            : int  — number of passengers that entered
- FASCIA_ETA         : str  — age bracket (e.g. '18-25', '26-40')
- GENERE             : str  — gender ('M' or 'F')
- TIPO_DOCUMENTO     : str  — document type (e.g. 'PASSAPORTO', 'CARTA DI IDENTITA')
- ESITO_CONTROLLO    : str  — check outcome ('OK', 'FERMATO', 'RESPINTO', 'SEGNALATO')
- ROTTA              : str  — route string e.g. 'IST → FCO'
"""

    prompt = f"""
You are a Python/pandas expert. Convert the user's natural language query into a
pandas boolean filter expression for an airport security dataset.

You have access to TWO dataframes: `df_allarmi` and `df_tipologia`.
Choose the most appropriate one based on the query.

{ALLARMI_SCHEMA}
{TIPOLOGIA_SCHEMA}

Rules:
1. Reply ONLY with a valid JSON object with exactly these keys:
   - "dataframe": either "allarmi" or "tipologia" (which df the filter applies to)
   - "expr": a valid pandas boolean expression as a string (will be passed to df.query())
   - "description": a short human-readable description of the filter (max 10 words)
2. The "expr" must use df.query() syntax (no df[] brackets, no variable names).
3. If NO filter is needed (e.g. "all data", "full dataset"), reply with:
   {{"dataframe": "allarmi", "expr": null, "description": "no filter, full dataset"}}
4. String comparisons must use double quotes inside the expr string.
5. No markdown, no backticks, no extra text outside the JSON.

Examples:
- "flights to Fiumicino"
  → {{"dataframe": "allarmi", "expr": "AREOPORTO_ARRIVO == \\"FCO\\"", "description": "arrivals at Fiumicino"}}

- "routes to Fiumicino with more than 10 alarmed"
  → {{"dataframe": "tipologia", "expr": "AREOPORTO_ARRIVO == \\"FCO\\" and ALLARMATI > 10", "description": "FCO arrivals, alarmed > 10"}}

- "flights from Albania in March"
  → {{"dataframe": "allarmi", "expr": "PAESE_PART == \\"ALBANIA\\" and MESE_PARTENZA == 3", "description": "Albania departures in March"}}

- "Turkish male passengers stopped"
  → {{"dataframe": "tipologia", "expr": "NAZIONALITA == \\"TURCO\\" and GENERE == \\"M\\" and ESITO_CONTROLLO == \\"FERMATO\\"", "description": "Turkish male passengers stopped"}}

- "top routes by alarms to MXP"
  → {{"dataframe": "tipologia", "expr": "AREOPORTO_ARRIVO == \\"MXP\\"", "description": "arrivals at Malpensa"}}

- "all data"
  → {{"dataframe": "allarmi", "expr": null, "description": "no filter, full dataset"}}

User query: "{user_query}"
"""

    raw = llm_json.invoke(prompt).content.strip()
    clean = raw.replace("```json", "").replace("```", "").strip()

    try:
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        print(f"WARNING: LLM returned invalid JSON for query '{user_query}'.")
        print(f"   Raw output: {raw!r}")
        print("   → Falling back to full dataset.")
        return None

    expr = parsed.get("expr")
    desc = parsed.get("description", "")
    target_df = parsed.get("dataframe", "allarmi")

    if not expr:
        if verbose:
            print(f"No filter extracted from '{user_query}' → full dataset.")
        return None

    if verbose:
        print(f"Query parsed: '{user_query}'")
        print(f"   Target df  : {target_df}")
        print(f"   Expression : {expr}")
        print(f"   Description: {desc}")

    # Return a special dict — DataAgent will detect and apply it
    return {"__pandas_expr__": expr, "__target_df__": target_df, "description": desc}

In [31]:
#change the query to filter the pipeline on specific criteria.
#set filters=None to run on the full dataset.

filters = {"AREOPORTO_PARTENZA": "IST"}

result = run_anomaly_detection_pipeline(
    filters=filters,
    contamination=0.1,
    z_threshold=2.5,
)

display(Markdown("##  Final Transit Anomaly Report"))
display(Markdown(result['report']['report']))

DataAgent — filters applied: {'AREOPORTO_PARTENZA': 'IST'}
   ALLARMI:      295 rows  (of 5080 total)
   TIPOLOGIA:    104 rows  (of 5095 total)
## Transit Anomaly Report – Filtered Dataset: IST

**1. Executive Summary:**

This report details transit anomalies identified within a filtered dataset focusing on departures from Rome Fiumicino Airport (IST). Analysis revealed a significant spike in alert rates and alarm occurrences on specific routes, primarily involving flights departing to Bermuda (BLQ), with a notable increase in rejected/stopped flights. Further investigation identified elevated activity on routes to Milan (MXP) and Rome Fiumicino (FCO), warranting closer scrutiny.

**2. HIGH RISK Anomalies:**

* **Route: IST → BLQ**
    * **Alert Rate:** 0.2941 | Alarmed: 5 | Entered: 17
    * **Severity:** 0.869 | Votes: 2/3 | Methods: LOF, Z-score
    * **Top Z-Score Feature:** `occ_Allarmi_generati` (Number of alarms generated)
    * **Reason:** This route exhibited a significantly 

##  Final Transit Anomaly Report

## Transit Anomaly Report – Filtered Dataset: IST

**1. Executive Summary:**

This report details transit anomalies identified within a filtered dataset focusing on departures from Rome Fiumicino Airport (IST). Analysis revealed a significant spike in alert rates and alarm occurrences on specific routes, primarily involving flights departing to Bermuda (BLQ), with a notable increase in rejected/stopped flights. Further investigation identified elevated activity on routes to Milan (MXP) and Rome Fiumicino (FCO), warranting closer scrutiny.

**2. HIGH RISK Anomalies:**

* **Route: IST → BLQ**
    * **Alert Rate:** 0.2941 | Alarmed: 5 | Entered: 17
    * **Severity:** 0.869 | Votes: 2/3 | Methods: LOF, Z-score
    * **Top Z-Score Feature:** `occ_Allarmi_generati` (Number of alarms generated)
    * **Reason:** This route exhibited a significantly lower alert rate than the global baseline (0.3543), with 6 rejected/stopped flights flagged. The high severity score (0.87) indicates a strong likelihood of an underlying issue requiring immediate investigation. The leading feature, `occ_Allarmi_generati`, suggests a potential surge in erroneous alerts or a change in data quality impacting this specific route.
    * **Recommended Action:** Conduct immediate investigation into the root cause of the increased alarm rate and rejected/stopped flights on this route.  Examine data sources contributing to `occ_Allarmi_generati` for inconsistencies and potential errors.  Consider implementing stricter validation rules for this route to mitigate future anomalies.


**3. MEDIUM RISK Anomalies:**

* **Route: IST → FCO**
    * **Alert Rate:** 0.3529 | Alarmed: 12 | Entered: 34
    * **Severity:** 0.722 | Votes: 2/3 | Methods: IsolationForest, Z-score
    * **Top Z-Score Feature:** `occ_Errata_segnalazione_SDI` (Error reporting related to SDI)
    * **Reason:** This route showed a moderate increase in alert rate (0.3529), resulting in 12 alarms and 34 entries. The severity score of 0.722 suggests a notable deviation from the baseline, though less pronounced than the BLQ route. The leading feature indicates potential issues with data integrity related to SDI (System for Digital Intelligence) reporting, requiring further investigation into the accuracy of this data source.
    * **Recommended Action:** Investigate potential errors within the SDI reporting process for this route. Cross-reference data with other sources to validate accuracy and identify any discrepancies.

* **Route: IST → MXP**
    * **Alert Rate:** 0.5000 | Alarmed: 25 | Entered: 50
    * **Severity:** 0.700 | Votes: 2/3 | Methods: IsolationForest, Z-score
    * **Top Z-Score Feature:** `occ_Errata_segnalazione_NSIS` (Error reporting related to NSIS)
    * **Reason:** This route demonstrated the highest alert rate within the medium risk category (0.5000), resulting in 25 alarms and 50 entries. The severity score of 0.700 indicates a significant deviation from the baseline.  The leading feature, `occ_Errata_segnalazione_NSIS`, points to potential errors in NSIS (National System for Identification and Security) reporting.
    * **Recommended Action:** Thoroughly investigate the NSIS data source for this route, focusing on identifying and correcting any errors or inconsistencies.  Analyze the nature of the NSIS reporting process to determine if changes are needed to improve data quality and reduce false positives.



**4. Overall Risk Assessment:**

The analysis highlights a concerning trend of elevated alert rates and alarm occurrences, particularly on routes departing from IST. The BLQ route presents the highest risk due to a substantial deviation from the baseline and a high number of rejected/stopped flights. The MXP and FCO routes, while exhibiting lower severity scores, still warrant attention due to their elevated alert rates and potential data integrity issues related to SDI and NSIS reporting, respectively.  Continuous monitoring of these routes and further investigation into the underlying causes are crucial to maintain data integrity and ensure accurate border control operations.  Further analysis should explore potential correlations between these anomalies and external factors (e.g., changes in regulations, seasonal travel patterns).

In [32]:
#change USER_QUERY to filter the pipeline on specific criteria.
#set USER_QUERY = "all data" to run on the full dataset.
USER_QUERY = "flights from Turkey with alarmed passengers greater than 10"

#step 1: LLM translates the query into filters
filters = parse_query_to_filters(USER_QUERY)

#step 2: run pipeline with extracted filters
result = run_anomaly_detection_pipeline(
    filters=filters,
    contamination=0.1,
    z_threshold=2.5,
)

display(Markdown("## Final Transit Anomaly Report"))
display(Markdown(f"**Query:** *{USER_QUERY}*"))
display(Markdown(f"**Filters applied:** `{filters}`"))
display(Markdown(result['report']['report']))

Query parsed: 'flights from Turkey with alarmed passengers greater than 10'
   Target df  : allarmi
   Expression : PAESE_PART == "TURCHIA" and ALLARMATI > 10
   Description: Flights from Turkey with alarmed passengers.
DataAgent — filters applied: {'__pandas_expr__': 'PAESE_PART == "TURCHIA" and ALLARMATI > 10', '__target_df__': 'allarmi', 'description': 'Flights from Turkey with alarmed passengers.'}
   ALLARMI:     5080 rows  (of 5080 total)
   TIPOLOGIA:   5095 rows  (of 5095 total)
Okay, here's a Transit Anomaly Report based on your specifications and the provided data. It aims to be professional, detailed, and actionable for border control analysts.

---

**TRANSIT ANOMALY REPORT – TURKEY FOCUS**

**DATE:** October 26, 2023
**ANALYSIS SCOPE:** Filtered dataset – {'__pandas_expr__': 'PAESE_PART == "TURCHIA" and ALLARMATI > 10', '__target_df__': 'allarmi', 'description': 'Flights from Turkey with alarmed passengers.'}

**EXECUTIVE SUMMARY:** This report identifies a cluster of high

## Final Transit Anomaly Report

**Query:** *flights from Turkey with alarmed passengers greater than 10*

**Filters applied:** `{'__pandas_expr__': 'PAESE_PART == "TURCHIA" and ALLARMATI > 10', '__target_df__': 'allarmi', 'description': 'Flights from Turkey with alarmed passengers.'}`

Okay, here's a Transit Anomaly Report based on your specifications and the provided data. It aims to be professional, detailed, and actionable for border control analysts.

---

**TRANSIT ANOMALY REPORT – TURKEY FOCUS**

**DATE:** October 26, 2023
**ANALYSIS SCOPE:** Filtered dataset – {'__pandas_expr__': 'PAESE_PART == "TURCHIA" and ALLARMATI > 10', '__target_df__': 'allarmi', 'description': 'Flights from Turkey with alarmed passengers.'}

**EXECUTIVE SUMMARY:** This report identifies a cluster of high-risk flight routes originating from Turkey exhibiting significantly elevated rates of passenger alarms compared to historical averages.  The analysis indicates a potential increase in security concerns related to travel from Turkey, requiring heightened scrutiny and further investigation.

**GLOBAL BASELINES:**
*   Avg alert rate: 0.2074
*   Std: 0.9736
*   P95: 1.0000
*   Avg alarms/route: 472.4

**RISK DISTRIBUTION:** HIGH=20, MEDIUM=65, LOW=25


**I. HIGH RISK ANOMALIES (15 Anomalies)**

| Route          | Alert Rate | Alarmed | Entered | Severity | Votes | Methods           | Top Z-Score Feature             | Reason                                                                 | Recommended Action                                                                |
|----------------|------------|--------|--------|----------|-------|-------------------|----------------------------------|---------------------------------------------------------------------------|------------------------------------------------------------------------------------|
| BHX → FCO       | 0.0000      | 0      | 1       | 0.429    | 3/3   | IsolationForest, LOF, Z-score | investigation_rate              | Rejected/Stopped: 1 – All 3 methods agree.                               | Investigate passenger manifests and travel itineraries for this route immediately.|
| TIA → RMI       | 0.1356      | 395    | 2913   | 0.310    | 3/3   | IsolationForest, LOF, Z-score | occ_Allarmi_generati             | Rejected/Stopped: 8 – All 3 methods agree.                                | Prioritize passenger screening and enhanced security protocols for this route.       |
| PVG → MXP       | 1.2000      | 6      | 5      | 0.241    | 2/3   | IsolationForest, Z-score | occ_Allarmi_generati_da_INTER   | Alert rate 5.8x baseline – Significant deviation.                         | Conduct a deeper dive into the nature of these alarms and potential illicit activity.|
| LHR → LIN       | 0.0588      | 1      | 17     | 0.417    | 2/3   | IsolationForest, Z-score | occ_Viaggiatori_entrati_nel_S    | Volume 13288 > 10201 – Rejected/Stopped: 6                               | Review passenger data and potential connections to organized crime.               |
| PVG → FCO       | 0.7500      | 6      | 8      | 0.164    | 2/3   | IsolationForest, Z-score | rolling_deviation             | Alert rate 3.6x baseline – Significant deviation.                         | Investigate potential connections and passenger backgrounds.                     |
| BFS → BGY       | 0.7000      | 7      | 10     | 0.162    | 2/3   | LOF, Z-score | occ_Allarmi_generati_da_INTER   | Alert rate 3.4x baseline – Significant deviation.                         |  Review passenger manifests and travel itineraries for this route immediately.|
| EVN → VCE       | 2.0000      | 2      | 1      | 0.226    | 2/3   | LOF, Z-score | occ_Nulla_a_procedere_NSIS     | Alert rate 9.6x baseline – Extreme deviation.                             | Immediate investigation into the reason for these alarms and potential fraud.     |
| IST → VCE       | 0.7826      | 18     | 23     | 0.237    | 2/3   | IsolationForest, Z-score | occ_Viaggiatori_con_Allarmi     | Alert rate 3.8x baseline – Significant deviation.                         |  Scrutinize passenger documentation and conduct enhanced security checks.          |
| RAK → CIA       | 1.0000      | 4      | 4      | 0.275    | 2/3   | LOF, Z-score | occ_Errata_segnalazione_NSIS    | Alert rate 4.8x baseline – Significant deviation.                         | Investigate potential discrepancies in passenger information and travel history.   |
| IST → NAP       | 0.6667      | 2      | 3      | 0.173    | 2/3   | IsolationForest, Z-score | occ_Allarmi_generati_da_SDI/N   | Alert rate 3.2x baseline – Significant deviation.                         |  Review passenger data and potential connections to organized crime.               |
| Lcy → Flr       | 0.0000      | 0      | 0      | 0.453    | 2/3   | LOF, Z-score | allarmi_tot_mean             | Volume 100000 > 10201 – Statistical anomaly.                             |  Investigate the reason for this unusually low volume and potential data issues.|
| TIA → VRN       | 0.6816      | 6944   | 10188  | 0.480    | 2/3   | IsolationForest, Z-score | allarmati_sum               | Alert rate 3.3x baseline – Significant deviation.                         |  Implement enhanced screening procedures for passengers arriving at VRN from TIA.|
| GRU → MXP       | 1.0000      | 2      | 2      | 0.290    | 2/3   | LOF, Z-score | occ_Altro                   | Alert rate 4.8x baseline – Significant deviation.                         | Investigate potential connections and passenger backgrounds.                     |
| CMN → BLQ       | 0.6667      | 2      | 3      | 0.264    | 2/3   | IsolationForest, Z-score | occ_Notifica_Atti/Provv        | Alert rate 3.2x baseline – Significant deviation.                         | Review passenger data and potential connections to organized crime.               |
| DOH → MXP       | 1.2000      | 6      | 5      | 0.512    | 2/3   | IsolationForest, Z-score | occ_???                    | Alert rate 5.8x baseline – Extreme deviation.                             | Immediate investigation into the reason for these alarms and potential illicit activity.|



**II. MEDIUM RISK ANOMALIES (65 Anomalies)**

*(Detailed list of Medium Risk anomalies – similar format as above, but with fewer entries)*

**III. LOW RISK ANOMALIES (25 Anomalies)**

*(Detailed list of Low Risk anomalies – similar format as above, but with fewer entries)*


**IV. OVERALL RISK ASSESSMENT:**

The analysis reveals a statistically significant increase in alarm rates originating from Turkey compared to historical norms.  While the majority of anomalies are classified as High Risk, a substantial number (65) fall into Medium and Low risk categories.  The presence of multiple detection methods agreeing on these anomalies strengthens the confidence in their validity and warrants immediate attention.  Continuous monitoring of these routes and further investigation into potential underlying causes (e.g., organized crime, human trafficking) are strongly recommended.

---

**Note:** This report is based on the provided data and analysis scope.  A more comprehensive assessment would require access to additional datasets, including passenger profiles, travel history, and intelligence information.  The "Top Z-score feature" is a simplified representation; a full analysis would involve more sophisticated feature engineering and interpretation.  The "Recommended Action" section should be adapted based on specific operational procedures and risk tolerance levels.